In [23]:
import math
import random

class Value:

    def __init__(self, data, children=()):
        self.data = data
        self.children = set(children)
        self.grad = 0.0
        self.backward = lambda : None

    def __add__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        out = Value(self.data + other.data, (self, other))
        def backward():
            self.grad = out.grad
            other.grad = out.grad

        out.backward = backward
        return out

    def __mul__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        out = Value(self.data * other.data, (self, other))
        def backward():
            self.grad = out.grad * other.data
            other.grad = out.grad * self.data

        out.backward = backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self, ))
        def backward():
            self.grad = (1 - t**2) * out.grad
            
        out.backward = backward
        return out

    def __repr__(self):
        return f'Value: {self.data:.f4} Grad: {self.grad:.4f}'

    def backwards(self):
        topo = []
        visited = set()
        def topo_sort(node):
            if node not in visited:
                visited.add(node)
                for child in node.children:
                    topo_sort(child)
                topo.append(node)

        self.grad = 1.0
        topo_sort(self)

        for node in reversed(topo):
            node.backward()

class Neuron:

    def __init__(self, nin):
        self.weights = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.bias = Value(random.uniform(-1, 1))
        self.parameters = [self.bias] + self.weights

    def __call__(self, x):
        act = sum([wi * xi  for wi, xi in zip(self.weights, x)], self.bias)
        return act.tanh()

class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
        self.parameters = []
        for n in self.neurons:
            for p in n.parameters:
                self.parameters.append(p)

    def __call__(self, x):
        return [n(x) for n in self.neurons]

class MLP:

    def __init__(self, nin, nouts):
        size = [nin] + nouts
        self.layers = [Layer(size[i], size[i + 1]) for i in range(len(nouts))]
        self.parameters = []
        for l in self.layers:
            for p in l.parameters:
                self.parameters.append(p)

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

model = MLP(784, [16, 16, 10])

len(model.parameters)
        

13002

In [22]:
import torch
import torch.nn as nn

class NN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 16)
        self.fc2 = nn.Linear(16, 16)
        self.out = nn.Linear(16, 10)

    def forward(self, x):
        return x

model2 = NN()
print(sum([p.numel() for p in model2.parameters()]))

13002
